# Quantitative Asset Forecasting — LSTM-Optimized Multivariate Pipeline

Raw multivariate OHLCV sequences fed directly into an attention-based LSTM.
No hand-engineered features — the model learns temporal patterns from data.

**Assets:** GLD · SPY · QQQ · AAPL  
**Architecture:** Multivariate LSTM with attention (all assets as joint input)  
**Target:** 5-day direction per asset  
**Speed optimizations:** reduced hidden size, single layer, early stopping, cached data


## Setup

In [1]:
# Uncomment if needed
# !pip install yfinance pandas numpy matplotlib scikit-learn statsmodels torch

## Imports and Configuration

In [2]:
import warnings
warnings.filterwarnings("ignore")

import os, math, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tools.sm_exceptions import ConvergenceWarning

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.simplefilter("ignore", ConvergenceWarning)
np.random.seed(42)
torch.manual_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ── CONFIG ──────────────────────────────────────────────────────────────────
TICKERS     = ["GLD", "SPY", "QQQ", "AAPL"]
START_DATE  = "2015-01-01"
END_DATE    = "2025-12-31"
CACHE_FILE  = "market_data_cache.pkl"

TRAIN_RATIO      = 0.75
SEQUENCE_LENGTH  = 30       # 30 days lookback — sweet spot for daily LSTM
HORIZON          = 5        # predict 5-day direction

# Speed-optimised training
EPOCHS              = 40
BATCH_SIZE          = 128   # larger batch = faster, still stable
LEARNING_RATE       = 3e-4
EARLY_STOP_PATIENCE = 12

# LSTM architecture — lean but capable
HIDDEN_SIZE  = 64
NUM_LAYERS   = 2
DROPOUT      = 0.2

ARIMA_REFIT_EVERY    = 10      # refit every N steps (faster than every step)
INITIAL_CAPITAL      = 10_000
CONFIDENCE_THRESHOLD = 0.60

# Raw OHLCV columns used per ticker (no manual feature engineering)
RAW_COLS = ["Open", "High", "Low", "Close", "Volume"]


Using device: cpu


## Data Retrieval with Caching

Downloads 10 years of OHLCV for all tickers jointly and caches locally.
On repeat runs this cell is instant.


In [3]:
def download_market_data(tickers, start, end):
    data = yf.download(tickers, start=start, end=end,
                       auto_adjust=True, progress=False)
    if data.empty:
        raise ValueError("No data downloaded.")
    return data

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "rb") as f:
        market_data = pickle.load(f)
    print("Loaded from cache.")
else:
    market_data = download_market_data(TICKERS, START_DATE, END_DATE)
    with open(CACHE_FILE, "wb") as f:
        pickle.dump(market_data, f)
    print("Downloaded and cached.")

print("Shape:", market_data.shape)
market_data.head()


Loaded from cache.
Shape: (2765, 20)


Price           Close                                          High  \
Ticker           AAPL         GLD        QQQ         SPY       AAPL   
Date                                                                  
2015-01-02  24.214893  114.080002  94.784431  170.589630  24.682226   
2015-01-05  23.532717  115.800003  93.394066  167.508850  24.064280   
2015-01-06  23.534931  117.120003  92.141808  165.931061  23.794068   
2015-01-07  23.864941  116.430000  93.329620  167.998749  23.964608   
2015-01-08  24.781893  115.940002  95.115929  170.979919  24.839479   

Price                                                Low              \
Ticker             GLD        QQQ         SPY       AAPL         GLD   
Date                                                                   
2015-01-02  114.800003  95.944601  171.793724  23.776353  112.320000   
2015-01-05  116.000000  94.480579  169.709412  23.346671  114.730003   
2015-01-06  117.500000  93.688707  168.339223  23.173911  115.800003   
2015-01-07  116.879997  93.550604  168.339217  23.632381  116.169998   
2015-01-08  116.870003  95.300081  171.195832  24.075357  115.849998   

Price                                   Open                         \
Ticker            QQQ         SPY       AAPL         GLD        QQQ   
Date                                                                  
2015-01-02  94.324045  169.551627  24.671151  112.489998  95.539465   
2015-01-05  93.127041  167.201605  23.984545  114.779999  94.370084   
2015-01-06  91.727462  165.133869  23.596947  116.220001  93.532178   
2015-01-07  92.528545  166.811279  23.743124  116.470001  92.749535   
2015-01-08  94.020206  169.393860  24.192745  116.449997  94.121491   

Price                      Volume                                 
Ticker             SPY       AAPL       GLD       QQQ        SPY  
Date                                                              
2015-01-02  171.378523  212818400   7109600  31314600  121465900  
2015-01-05  169.543334  257142000   8177400  36521300  169632600  
2015-01-06  167.816066  263188400  11238300  66205500  209151400  
2015-01-07  167.259691  160423600   6434200  37577400  125346700  
2015-01-08  169.410459  237458000   7033700  40212600  147217800

## Multivariate Sequence Builder

**Key design choice:** instead of engineering features per ticker, we stack raw
OHLCV from *all four assets* into a single wide matrix. Each timestep has
`len(TICKERS) × len(RAW_COLS)` = 20 input features.

This lets the LSTM learn cross-asset dependencies (e.g. GLD reacting to SPY
moves) that XGBoost on single-ticker features cannot capture.

The target for each sample is the 5-day direction of the *focal ticker*.


In [4]:
def build_multivariate_dataset(market_df, focal_ticker, tickers=TICKERS,
                               raw_cols=RAW_COLS, horizon=HORIZON):
    """
    Returns a DataFrame where:
      - columns = all raw OHLCV for every ticker (20 cols)
      - target  = 1 if focal_ticker Close rises over next HORIZON days, else 0
    All rows with any NaN are dropped.
    """
    frames = []
    for t in tickers:
        sub = pd.DataFrame(
            {f"{t}_{c}": market_df[c][t] for c in raw_cols}
        )
        frames.append(sub)

    df = pd.concat(frames, axis=1).dropna().copy()

    focal_close = df[f"{focal_ticker}_Close"]
    df["target"] = (focal_close.shift(-horizon) > focal_close).astype(int)

    # drop last HORIZON rows (no valid target)
    df = df.iloc[:-horizon]
    df = df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

    if len(df) < 300:
        raise ValueError(f"Not enough rows for {focal_ticker}: {len(df)}")

    return df


# Quick sanity check
example = build_multivariate_dataset(market_data, "GLD")
print("Shape:", example.shape)
print("Target balance (% up):", example["target"].mean().round(3))
example.head()


Shape: (2760, 21)
Target balance (% up): 0.549


,GLD_Open,GLD_High,GLD_Low,GLD_Close,GLD_Volume,SPY_Open,SPY_High,SPY_Low,SPY_Close,SPY_Volume,...,QQQ_High,QQQ_Low,QQQ_Close,QQQ_Volume,AAPL_Open,AAPL_High,AAPL_Low,AAPL_Close,AAPL_Volume,target
0,112.489998,114.800003,112.320000,114.080002,7109600,171.378523,171.793724,169.551627,170.589630,121465900,...,95.944601,94.324045,94.784431,31314600,24.671151,24.682226,23.776353,24.214893,212818400,1
1,114.779999,116.000000,114.730003,115.800003,8177400,169.543334,169.709412,167.201605,167.508850,169632600,...,94.480579,93.127041,93.394066,36521300,23.984545,24.064280,23.346671,23.532717,257142000,1
2,116.220001,117.500000,115.800003,117.120003,11238300,167.816066,168.339223,165.133869,165.931061,209151400,...,93.688707,91.727462,92.141808,66205500,23.596947,23.794068,23.173911,23.534931,263188400,1
3,116.470001,116.879997,116.169998,116.430000,6434200,167.259691,168.339217,166.811279,167.998749,125346700,...,93.550604,92.528545,93.329620,37577400,23.743124,23.964608,23.632381,23.864941,160423600,1
4,116.449997,116.870003,115.849998,115.940002,7033700,169.410459,171.195832,169.393860,170.979919,147217800,...,95.300081,94.020206,95.115929,40212600,24.192745,24.839479,24.075357,24.781893,237458000,1


## Sequence Builder

In [5]:
def create_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(seq_len, len(X)):
        Xs.append(X[i - seq_len:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)


## Attention-Based Multivariate LSTM

Two improvements over a vanilla LSTM:

1. **Attention over timesteps** — the model learns *which days* in the 30-day
   window matter most, rather than relying only on the final hidden state.
2. **LayerNorm** on the LSTM output — stabilises training and reduces the need
   for large dropout, which helps on moderate-sized datasets.


In [6]:
class MultivariateLSTMAttention(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True
        )
        self.norm      = nn.LayerNorm(hidden_size)
        self.attention = nn.Linear(hidden_size, 1)
        self.fc        = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _       = self.lstm(x)            # (B, T, H)
        out          = self.norm(out)
        attn_w       = torch.softmax(self.attention(out), dim=1)  # (B, T, 1)
        context      = (attn_w * out).sum(dim=1)                  # (B, H)
        return self.fc(context)                                    # (B, 1)


## Training Function

Speed optimisations applied:
- `BATCH_SIZE = 128` (2× faster than 64 with negligible accuracy cost)
- `ReduceLROnPlateau` with aggressive factor=0.5 to converge faster
- Early stopping with patience=12
- Label smoothing (0.1) — prevents overconfident predictions on noisy labels
- Plain `BCEWithLogitsLoss` (no pos_weight) — markets are ~50/50 up/down


In [7]:
def train_and_predict(df, split_idx,
                      seq_len=SEQUENCE_LENGTH, epochs=EPOCHS,
                      batch_size=BATCH_SIZE, lr=LEARNING_RATE):

    feature_cols = [c for c in df.columns if c != "target"]
    X_raw = df[feature_cols].values.astype(np.float32)
    y_raw = df["target"].values.astype(np.float32)

    X_tr, X_te = X_raw[:split_idx], X_raw[split_idx:]
    y_tr, y_te = y_raw[:split_idx], y_raw[split_idx:]

    # Scale features — fit only on train to avoid leakage
    scaler     = MinMaxScaler()
    X_tr_sc    = scaler.fit_transform(X_tr)
    X_te_sc    = scaler.transform(X_te)

    X_tr_seq, y_tr_seq = create_sequences(X_tr_sc, y_tr, seq_len)
    X_te_seq, y_te_seq = create_sequences(X_te_sc, y_te, seq_len)

    if len(X_tr_seq) < 20 or len(X_te_seq) < 10:
        raise ValueError("Not enough sequences.")

    # Carve 10% of train as validation
    val_n        = max(1, int(0.1 * len(X_tr_seq)))
    X_val_seq    = X_tr_seq[-val_n:]
    y_val_seq    = y_tr_seq[-val_n:]
    X_tr_seq2    = X_tr_seq[:-val_n]
    y_tr_seq2    = y_tr_seq[:-val_n]

    # Label smoothing
    y_tr_smooth  = y_tr_seq2 * 0.8 + 0.1

    X_tr_t  = torch.tensor(X_tr_seq2,  dtype=torch.float32)
    y_tr_t  = torch.tensor(y_tr_smooth.reshape(-1, 1), dtype=torch.float32)
    X_val_t = torch.tensor(X_val_seq,  dtype=torch.float32)
    y_val_t = torch.tensor(y_val_seq.reshape(-1, 1),   dtype=torch.float32)
    X_te_t  = torch.tensor(X_te_seq,   dtype=torch.float32)

    loader = DataLoader(TensorDataset(X_tr_t, y_tr_t),
                        batch_size=batch_size, shuffle=False)

    model     = MultivariateLSTMAttention(
        input_size  = X_tr_t.shape[2],
        hidden_size = HIDDEN_SIZE,
        num_layers  = NUM_LAYERS,
        dropout     = DROPOUT
    ).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=5, factor=0.5)

    best_loss, best_state, patience_ctr = float("inf"), None, 0

    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(
                model(X_val_t.to(DEVICE)),
                y_val_t.to(DEVICE)
            ).item()

        scheduler.step(val_loss)

        if val_loss < best_loss:
            best_loss, best_state, patience_ctr = val_loss, model.state_dict(), 0
        else:
            patience_ctr += 1
        if patience_ctr >= EARLY_STOP_PATIENCE:
            break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        logits     = model(X_te_t.to(DEVICE)).cpu().numpy().flatten()
    probs          = torch.sigmoid(torch.tensor(logits)).numpy()
    pred           = (probs >= 0.5).astype(int)
    actual         = y_te_seq.flatten().astype(int)

    return pred, actual, probs

## Metrics and Backtesting Helpers

In [8]:
def sharpe_ratio(returns, periods=252):
    r = np.asarray(returns)
    return 0.0 if r.std() == 0 else (r.mean() / r.std()) * np.sqrt(periods)

def max_drawdown(portfolio):
    pv    = np.asarray(portfolio)
    peaks = np.maximum.accumulate(pv)
    return ((pv - peaks) / peaks).min()

def strategy_metrics(prev_close, actual_next, pred_direction, capital=10_000):
    sig      = np.asarray(pred_direction).astype(int)
    ret_act  = (np.asarray(actual_next) - np.asarray(prev_close)) / np.asarray(prev_close)
    strat_r  = sig * ret_act
    bh_r     = ret_act.copy()
    strat_pv = capital * np.cumprod(1 + strat_r)
    bh_pv    = capital * np.cumprod(1 + bh_r)
    return {
        "strategy_cum_return_pct":  round((strat_pv[-1] / capital - 1) * 100, 2),
        "buyhold_cum_return_pct":   round((bh_pv[-1]    / capital - 1) * 100, 2),
        "sharpe_ratio":             round(sharpe_ratio(strat_r), 3),
        "max_drawdown_pct":         round(max_drawdown(strat_pv) * 100, 2),
    }

def arima_direction(train_series, test_close, order=(1,1,1), refit_every=ARIMA_REFIT_EVERY, horizon=5):
    """Walk-forward ARIMA predicting HORIZON-step-ahead price, converted to direction."""
    history = list(train_series)
    preds   = []
    fit     = None
    for i, cp in enumerate(test_close):
        if i % refit_every == 0 or fit is None:
            try:
                fit = ARIMA(history, order=order,
                            enforce_stationarity=False,
                            enforce_invertibility=False).fit(
                    method_kwargs={"warn_convergence": False})
            except Exception:
                fit = None
        try:
            fc = fit.forecast(steps=horizon)[-1] if fit else cp
        except Exception:
            fc = cp
        preds.append(int(fc > cp))
        history.append(cp)
    return np.array(preds)


## Per-Ticker Evaluation

For each ticker:
1. Build multivariate dataset (all 4 assets as input)
2. Train LSTM on train split
3. Compare against Naive and ARIMA baselines
4. Report directional accuracy, strategy return, Sharpe, drawdown
5. Add high-confidence filtered row (signals where model is >60% sure)


In [9]:
def evaluate_ticker(market_df, focal_ticker):
    print(f"[{focal_ticker}] building dataset...", flush=True)
    df         = build_multivariate_dataset(market_df, focal_ticker)
    split_idx  = int(len(df) * TRAIN_RATIO)

    focal_close      = df[f"{focal_ticker}_Close"].values
    train_close      = focal_close[:split_idx]
    test_close       = focal_close[split_idx:]

    # 5-day-ahead actual close for backtesting
    focal_close_full = market_df["Close"][focal_ticker].dropna().values
    # align: test window starts at split_idx in the feature df
    # next_close = close HORIZON days later
    test_next_close  = np.array([
        focal_close_full[split_idx + i + HORIZON]
        if split_idx + i + HORIZON < len(focal_close_full)
        else focal_close_full[-1]
        for i in range(len(test_close))
    ])

    # ── LSTM ──────────────────────────────────────────────────────────────
    print(f"[{focal_ticker}] training LSTM...", flush=True)
    lstm_pred, lstm_actual, lstm_probs = train_and_predict(df, split_idx)

    eval_len        = len(lstm_actual)
    seq             = SEQUENCE_LENGTH
    prev_c_eval     = test_close[seq:seq + eval_len]
    next_c_eval     = test_next_close[seq:seq + eval_len]

    # ── Baselines ─────────────────────────────────────────────────────────
    print(f"[{focal_ticker}] ARIMA baseline...", flush=True)
    arima_dir  = arima_direction(train_close, test_close, horizon=HORIZON)
    arima_eval = arima_dir[seq:seq + eval_len]

    # Naive: yesterday went up → predict up
    naive_dir  = (np.diff(test_close, prepend=train_close[-1]) > 0).astype(int)
    naive_eval = naive_dir[seq:seq + eval_len]

    rows = []

    for name, pred in [("Naive", naive_eval),
                        ("ARIMA", arima_eval),
                        ("LSTM_Multivariate", lstm_pred)]:
        acc   = accuracy_score(lstm_actual, pred) * 100
        strat = strategy_metrics(prev_c_eval, next_c_eval, pred)
        row   = {"Ticker": focal_ticker, "Model": name,
                 "Directional_Accuracy_%": round(acc, 2),
                 "Coverage_%": 100.0, **strat}
        rows.append(row)

    # High-confidence LSTM
    hc_mask = (lstm_probs >= CONFIDENCE_THRESHOLD) | (lstm_probs <= 1 - CONFIDENCE_THRESHOLD)
    if hc_mask.sum() > 10:
        hc_acc   = accuracy_score(lstm_actual[hc_mask], lstm_pred[hc_mask]) * 100
        hc_strat = strategy_metrics(prev_c_eval[hc_mask], next_c_eval[hc_mask],
                                    lstm_pred[hc_mask])
        rows.append({
            "Ticker": focal_ticker,
            "Model": f"LSTM_HighConf_{CONFIDENCE_THRESHOLD}",
            "Directional_Accuracy_%": round(hc_acc, 2),
            "Coverage_%": round(hc_mask.mean() * 100, 1),
            **hc_strat
        })

    print(f"[{focal_ticker}] done.", flush=True)
    return pd.DataFrame(rows)


## Run All Tickers (Sequential — safe in Jupyter)

In [10]:
all_results = []

for ticker in TICKERS:
    try:
        print(f"\n===== {ticker} =====", flush=True)
        result = evaluate_ticker(market_data, ticker)
        all_results.append(result)
        print(result.to_string(index=False))
    except Exception as e:
        print(f"Error for {ticker}: {e}")

final_results = pd.concat(all_results, ignore_index=True)
final_results



===== GLD =====
[GLD] building dataset...
[GLD] training LSTM...
[GLD] ARIMA baseline...
[GLD] done.
Ticker             Model  Directional_Accuracy_%  Coverage_%  strategy_cum_return_pct  buyhold_cum_return_pct  sharpe_ratio  max_drawdown_pct
   GLD             Naive                   49.85       100.0                   587.79                 4723.05         2.876            -24.70
   GLD             ARIMA                   46.97       100.0                   320.91                 4723.05         2.731            -30.78
   GLD LSTM_Multivariate                   43.79       100.0                   184.19                 4723.05         1.990            -40.44

===== SPY =====
[SPY] building dataset...
[SPY] training LSTM...
[SPY] ARIMA baseline...
[SPY] done.
Ticker             Model  Directional_Accuracy_%  Coverage_%  strategy_cum_return_pct  buyhold_cum_return_pct  sharpe_ratio  max_drawdown_pct
   SPY             Naive                   51.36       100.0                   230.53 

,Ticker,Model,Directional_Accuracy_%,Coverage_%,strategy_cum_return_pct,buyhold_cum_return_pct,sharpe_ratio,max_drawdown_pct
0,GLD,Naive,49.85,100.0,587.79,4723.05,2.876,-24.70
1,GLD,ARIMA,46.97,100.0,320.91,4723.05,2.731,-30.78
2,GLD,LSTM_Multivariate,43.79,100.0,184.19,4723.05,1.990,-40.44
3,SPY,Naive,51.36,100.0,230.53,1459.30,2.068,-45.59
4,SPY,ARIMA,48.64,100.0,276.19,1459.30,2.160,-42.46
5,SPY,LSTM_Multivariate,50.61,100.0,379.18,1459.30,3.523,-21.01
6,SPY,LSTM_HighConf_0.6,66.86,25.6,139.82,145.11,6.507,-19.10
7,QQQ,Naive,50.61,100.0,369.14,2715.60,2.122,-51.55
8,QQQ,ARIMA,51.21,100.0,494.94,2715.60,2.343,-48.71
9,QQQ,LSTM_Multivariate,40.00,100.0,25.34,2715.60,0.977,-20.57


In [ ]:
all_results = []

for ticker in TICKERS:
    try:
        print(f"\n===== {ticker} =====", flush=True)
        result = evaluate_ticker(market_data, ticker)
        all_results.append(result)
        print(result.to_string(index=False))
    except Exception as e:
        print(f"Error for {ticker}: {e}")

final_results = pd.concat(all_results, ignore_index=True)
final_results


===== GLD =====
[GLD] building dataset...
[GLD] training LSTM...
[GLD] ARIMA baseline...
[GLD] done.
Ticker             Model  Directional_Accuracy_%  Coverage_%  strategy_cum_return_pct  buyhold_cum_return_pct  sharpe_ratio  max_drawdown_pct
   GLD             Naive                   49.85       100.0                   587.79                 4723.05         2.876            -24.70
   GLD             ARIMA                   46.97       100.0                   320.91                 4723.05         2.731            -30.78
   GLD LSTM_Multivariate                   51.82       100.0                   655.43                 4723.05         3.409            -40.44

===== SPY =====
[SPY] building dataset...
[SPY] training LSTM...
[SPY] ARIMA baseline...
[SPY] done.
Ticker             Model  Directional_Accuracy_%  Coverage_%  strategy_cum_return_pct  buyhold_cum_return_pct  sharpe_ratio  max_drawdown_pct
   SPY             Naive                   51.36       100.0                   230.53 

In [ ]:
all_results = []

for ticker in TICKERS:
    try:
        print(f"\n===== {ticker} =====", flush=True)
        result = evaluate_ticker(market_data, ticker)
        all_results.append(result)
        print(result.to_string(index=False))
    except Exception as e:
        print(f"Error for {ticker}: {e}")

final_results = pd.concat(all_results, ignore_index=True)
final_results

## Save Results

In [ ]:
final_results.to_csv("quant_lstm_multivariate_results.csv", index=False)
print("Saved.")
final_results


## What Makes This LSTM-Appropriate

| Design choice | Why it helps LSTM specifically |
|---|---|
| Raw OHLCV sequences, no feature engineering | LSTM learns its own temporal patterns |
| All 4 assets as joint input | LSTM captures cross-asset dependencies XGBoost can't |
| 30-day lookback window | Enough context without drowning in noise |
| 5-day horizon target | Smoother signal than next-day, more learnable |
| LayerNorm on LSTM output | Stabilises gradients, faster convergence |
| Attention over timesteps | Model weights recent vs older days dynamically |
| Label smoothing | Prevents overconfidence on noisy binary labels |
| Large batch (128) + early stopping | Fast training without sacrificing stability |

## Reproducibility
- Data cached locally after first download
- Random seeds fixed (numpy + torch)
- No external CSV files required
